# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamed-6513/flyrank_ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

We use our trained Logistic Regression model to predict the probability of decline for all active pages in our dataset. We then sort the entire portfolio to find the pages at the highest risk.

To align with the business constraint defined in Week 2 (`Precision@50`), we isolate the **Top 50** highest-risk pages to create a manageable queue for our SEO editors. We provide the raw feature values as "reason codes" so editors understand *why* the model flagged the page (e.g., high age, low recent sessions).

In [6]:
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Load Data
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = 'https://raw.githubusercontent.com/mohamed-6513/flyrank_ml/main/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(csv_path)

# 2. Filter and prep (Train on everything with >100 impressions to build the best model)
df_active = df[df['impressions_prev_30d'] > 100].copy()
df_active['is_declining_label'] = (df_active['trend_direction'] == 'down').astype(int)

features = ['content_age_days', 'competition', 'impressions_prev_30d', 'search_volume', 'sessions_prev_30d']
target = 'is_declining_label'

# 3. Train final model on all active data
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000))
])
model.fit(df_active[features], df_active[target])

# 4. Predict and Rank
df_active['decline_probability'] = model.predict_proba(df_active[features])[:, 1]
df_ranked = df_active.sort_values(by='decline_probability', ascending=False)

# 5. Take top 50
action_queue = df_ranked.head(50).copy()

print(f"Action Queue Generated: {len(action_queue)} pages.")
print("Top 50 Highest Risk Pages (with Reason Codes):")
pd.set_option('display.max_rows', 50)
display(action_queue[['content_id', 'decline_probability'] + features])

Action Queue Generated: 50 pages.
Top 50 Highest Risk Pages (with Reason Codes):


,content_id,decline_probability,content_age_days,competition,impressions_prev_30d,search_volume,sessions_prev_30d
6903,content_c84a0ab98e90,0.697151,95,0.00,84773,0.0,19
482,content_39881853ef0c,0.690224,97,0.02,64917,170.0,11
27178,content_453722754fea,0.677154,97,0.00,32872,10.0,1
12370,content_8d3971bfd976,0.674098,95,0.00,54506,0.0,17
22028,content_73c54f78c06a,0.670555,97,0.00,97200,10.0,44
11791,content_809242db11fd,0.666573,90,1.00,611,10.0,0
6735,content_786aacb2f787,0.666341,90,1.00,348,10.0,0
17163,content_ec4906827f42,0.666227,90,1.00,219,10.0,0
3012,content_ed3690070bd2,0.666155,90,1.00,137,10.0,0
7510,content_a64c3a6b4e81,0.665465,90,1.00,1029,10.0,1


## 2. Intended use and limits

This list is a **directional decision-support tool** designed for SEO editors. It highlights the 50 pages mathematically most likely to decline so editors know where to focus their manual reviews.

**Limits:**
It is *not* an automated deletion or redirection tool. The model evaluates a 90-day snapshot and identifies correlations, but it does not guarantee causality.

In [7]:
print("Intended Use: Directional decision-support for manual SEO review.")

Intended Use: Directional decision-support for manual SEO review.


## 3. Human review + the no-go list

**Human Review Required:**
Editors must manually verify the page content is actually stale or irrelevant before making any edits.

**The No-Go List:**
Never alter the following types of pages, regardless of what the model scores them:
- Legal pages (Terms of Service, Privacy Policy)
- Homepages
- Contact pages

In [8]:
print("No-Go List: Legal pages, Homepages, Contact pages.")

No-Go List: Legal pages, Homepages, Contact pages.


## 4. Monitoring / retrain triggers

This model is a snapshot in time. It must be retrained under the following conditions:
1. **Base Rate Shift:** If the baseline percentage of declining pages shifts by more than 15% from the training data.
2. **Algorithm Update:** Automatically every 6 months to account for natural search engine algorithm drift.

In [9]:
print("Retrain Trigger: >15% Base Rate shift OR 6 months elapsed.")

Retrain Trigger: >15% Base Rate shift OR 6 months elapsed.


## 5. Exports for the paper

The final action queue (Top 50 pages) is exported below to `outputs/action_queue.csv` for use by the SEO team and inclusion in the final capstone paper.

In [10]:
# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

# Export the top 50
export_path = '../outputs/action_queue.csv'
action_queue.to_csv(export_path, index=False)

print(f"Successfully exported {len(action_queue)} pages to {export_path}")

Successfully exported 50 pages to ../outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.